# Bifurcation — segment metrics vs β, classical and quantum

Effect of the fork penalty on segment efficiency / purity / false-rate /
separation (AUC), for both forms, with **recomputed** metrics (A has changed).
Headline: classically benign but blunt; quantumly it **breaks the 1BQF**.

In [1]:
import sys; sys.path.insert(0,"/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Bifurification")
from pathlib import Path
import numpy as np, pandas as pd, pickle, time
import matplotlib.pyplot as plt
import bif
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT=Path(bif.__file__).resolve().parent/"outputs"; OUT.mkdir(parents=True,exist_ok=True)
TAU0=bif.threshold(0,'off')   # 0.35

## 1. Classical metrics vs β (dense fork, T ≤ 100)
True and false segments carry the same fork degree, so β **down-scales the whole
solution roughly uniformly**. At a **fixed τ = 0.35** efficiency *and* false-rate
collapse together (a threshold artefact); the **separation (AUC) is preserved**,
and the median true/false **ratio** is essentially unchanged.

In [2]:
BETAS=[0,0.005,0.01,0.02,0.05,0.1]
rows=[]
for T in [20,50,100]:
    ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr()
    B=bif.fork_graph(ham._segment_to_hit_ids); truth=bif.truth_mask(ev)
    for beta in BETAS:                       # dense off-diag fork
        A,b,diag,tau=bif.bif_system(A0,B,beta,'off')
        sol=bif.solve_classical(A,b); m=bif.metrics(sol,truth,TAU0)
        rows.append(dict(T=T,beta=beta,eff=m['segment_efficiency'],far=m['segment_false_rate'],
                         auc=bif.auc(sol,truth),
                         medT=np.median(sol[truth]),medF=np.median(sol[~truth])))
C=pd.DataFrame(rows)
display(C[C["T"]==50][['beta','eff','far','auc','medT','medF']].round(3))

fig,ax=plt.subplots(1,3,figsize=(15,4.6))
for T,mk in [(20,'o'),(50,'s'),(100,'^')]:
    d=C[C["T"]==T]
    ax[0].plot(d.beta,d.eff,mk+'-',color="#1b7837",label=f"T={T}")
    ax[0].plot(d.beta,d.far,mk+'--',color="#c51b7d")
    ax[1].plot(d.beta,d.auc,mk+'-',color="#2166ac",label=f"T={T}")
    ax[2].plot(d.beta,d.medT,mk+'-',color="#1b7837",label=f"T={T}")
    ax[2].plot(d.beta,d.medF,mk+'--',color="#c51b7d")
ax[0].set_title("(a) fixed τ=0.35: eff (—) & far (--) collapse together",fontweight="bold")
ax[0].axhline(TAU0,color="k",ls=":",lw=1); ax[0].set_ylabel("efficiency / false-rate")
ax[1].set_title("(b) AUC(true:false) preserved ≈ 1",fontweight="bold"); ax[1].set_ylim(0.4,1.02); ax[1].set_ylabel("AUC")
ax[2].set_title("(c) median true (—) & false (--) down-scale together",fontweight="bold")
ax[2].axhline(TAU0,color="k",ls=":",lw=1); ax[2].set_ylabel("median activation")
for a in ax: a.set_xlabel("β (dense off-diag fork)"); a.legend(fontsize=8)
fig.tight_layout()
for e,dp in (("pdf",600),("png",150)): fig.savefig(OUT/f"classical_metrics_vs_beta.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved classical_metrics_vs_beta")

,beta,eff,far,auc,medT,medF
6,0.000,1.00,0.02,1.0,0.423,0.250
7,0.005,0.50,0.00,1.0,0.377,0.222
8,0.010,0.50,0.00,1.0,0.340,0.200
9,0.020,0.02,0.00,1.0,0.284,0.167
10,0.050,0.00,0.00,1.0,0.191,0.111
11,0.100,0.00,0.00,1.0,0.125,0.072


saved classical_metrics_vs_beta


## 2. Classical vs quantum — the 1BQF breaks (T = 10)
The 1BQF base run relies on the false bulk sitting **on the notch** (erased). The
fork term lifts it off, so the one-bit filter re-promotes false segments: the
**quantum** AUC collapses to ≈ 0.5 while the **classical** AUC stays 1 — and the
dense fork circuit makes each quantum solve ~20× slower. (Cached; delete
`outputs/quantum_cache.pkl` to recompute. ~5–8 min.)

In [3]:
cache=OUT/"quantum_cache.pkl"
QBETAS=[0,0.005,0.01,0.02]
if cache.exists():
    qrows=pickle.load(open(cache,"rb"))
else:
    qrows=[]
    T=10; ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr()
    B=bif.fork_graph(ham._segment_to_hit_ids); truth=bif.truth_mask(ev)
    for beta in QBETAS:
        A,b,diag,tau=bif.bif_system(A0,B,beta,'off')
        solC=bif.solve_classical(A,b)
        t0=time.time(); solQ,panc,nsys=bif.solve_quantum(A,b,solC,tau); dt=time.time()-t0
        qrows.append(dict(beta=beta, nnz=int(A0.nnz+B.nnz), t_quantum=dt,
            auc_C=bif.auc(solC,truth), auc_Q=bif.auc(solQ,truth),
            far_C=bif.metrics(solC,truth,tau)['segment_false_rate'],
            far_Q=bif.metrics(solQ,truth,tau)['segment_false_rate']))
        print(f"  beta={beta}: AUC_C={qrows[-1]['auc_C']:.3f} AUC_Q={qrows[-1]['auc_Q']:.3f} "
              f"far_Q={qrows[-1]['far_Q']:.3f} t={dt:.0f}s")
    pickle.dump(qrows,open(cache,"wb"))
Q=pd.DataFrame(qrows); display(Q.round(3))

fig,ax=plt.subplots(1,2,figsize=(13,4.8))
ax[0].plot(Q.beta,Q.auc_C,'o-',color="#1b7837",lw=2,label="classical AUC")
ax[0].plot(Q.beta,Q.auc_Q,'s-',color="#d6604d",lw=2,label="quantum 1BQF AUC")
ax[0].axhline(0.5,color="k",ls=":",lw=1,label="random (0.5)")
ax[0].set_ylim(0.4,1.05); ax[0].set_xlabel("β (off-diag, T=10)"); ax[0].set_ylabel("AUC(true:false)")
ax[0].set_title("(a) Quantum discrimination collapses; classical intact",fontweight="bold"); ax[0].legend(fontsize=9)
ax[1].plot(Q.beta,Q.t_quantum,'D-',color="#6a3d9a",lw=2)
ax[1].set_xlabel("β (off-diag, T=10)"); ax[1].set_ylabel("1BQF solve time (s)")
ax[1].set_title("(b) Dense fork circuit: ~20× slower at β>0",fontweight="bold")
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"classical_vs_quantum_beta.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved classical_vs_quantum_beta")

,beta,nnz,t_quantum,auc_C,auc_Q,far_C,far_Q
0,0.000,7660,4.505,1.0,1.000,0.0,0.000
1,0.005,7660,86.326,1.0,0.547,0.0,0.927
2,0.010,7660,87.371,1.0,0.547,0.0,0.908
3,0.020,7660,89.442,1.0,0.547,0.0,0.922


saved classical_vs_quantum_beta


## 3. Summary

- **Classical, fixed τ:** efficiency and false-rate fall together — the fork term
  is a near-uniform down-scaling (true and false carry equal fork degree), so a
  fixed cut is the wrong readout.
- **Classical, β-aware τ / AUC:** the separation is essentially unchanged (AUC ≈ 1)
  — the fork penalty buys little extra separability on these clean low-T events
  and does **not** preferentially suppress the cross-track bridges (which look like
  true tracks to it).
- **Quantum (1BQF):** the term is actively harmful — it lifts the false bulk off
  the notch (so the one-bit filter re-promotes it → AUC ≈ 0.5) and breaks the
  sparse-A invariant (~20× slower). The 1BQF needs the false population pinned on
  the notch; the fork penalty removes exactly that.

See `bifurcation_hamiltonian.md` for the derivation and the analytic small-cluster
examples behind these curves.